# 01 · Build the training dataset

Streams RAID and DACTYL, draws a balanced sample stratified by generator,
domain and label, extracts the 30 features, and writes two Parquet files.

**Two datasets, because there are two models (PRD 8.1).**

| | rows | adversarial |
|---|---|---|
| `train_a.parquet` | Model A, the strict detector | included, ~15% of the mix |
| `train_b.parquet` | Model B, the surrogate | excluded entirely |

Excluding the adversarial rows from B is not an oversight — it is what makes B
a faithful stand-in for third-party detectors, which have no such hardening.

**The labelling rule that matters (E8).** RAID's paraphrase / synonym /
homoglyph variants are the humanized-text augmentation, free. Humanized *AI*
text stays labelled AI. Humanized *human* text stays labelled **human** —
labelling a rewritten human essay as AI would teach the model that editing is
evidence of machine authorship, which is exactly how detectors end up
punishing careful writers and non-native speakers (R2, A4).

Both datasets stream, so RAID's 16.7 GB never touches Kaggle's 20 GB disk.

Runtime: minutes on a smoke test, 1–2 hours at 400k rows. **CPU only — turn
the accelerator off for this stage to save GPU quota.**


In [ ]:
# Setup. Run once per session — everything below depends on it.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow google-api-python-client google-auth

import sys, os
from pathlib import Path

# Must be set before torch is imported anywhere — PyTorch reads it when CUDA
# first initialises. Reduces the fragmentation that turns "enough memory" into
# an out-of-memory error hours into a run.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Change this if you forked the repo. Public repo => no token needed.
GIT_URL = "https://github.com/ByteCraft-9/ai-text-humanizer.git"

# Either attached as a Kaggle Dataset named `ai-detector-repo`, or cloned.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

# parents=True so this also works off-Kaggle (Colab, a local box) after
# pointing WORK somewhere that exists.
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(parents=True, exist_ok=True)

print("repo:", REPO)
print("work:", WORK)
assert (REPO / "training" / "lib").is_dir(), "repo not found — check GIT_URL"


In [ ]:
# ---------------------------------------------------------------------------
# Run configuration. Read this cell before starting anything else.
# ---------------------------------------------------------------------------
#
# Kaggle gives 30 GPU-hours a week. A full run is 8-16 of them, so you cannot
# afford to discover a bug at hour six. Leave SMOKE_TEST = True for the first
# pass: it runs the entire pipeline end to end in well under an hour on a
# tiny sample. If stage 4 completes, the chain works. Then set it False and
# run for real.

SMOKE_TEST = True

if SMOKE_TEST:
    SAMPLE_ROWS = 5_000     # rows per dataset
    EPOCHS = 1
else:
    SAMPLE_ROWS = 400_000   # PRD 12.2
    EPOCHS = 3

print(f"{'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}: "
      f"{SAMPLE_ROWS:,} rows/dataset, {EPOCHS} epoch(s)")
if not SMOKE_TEST:
    print("Expect ~1-2 h for stage 1, then 4-8 h per model. Use "
          "Save Version -> Save & Run All so a browser disconnect cannot kill it.")


In [ ]:
# ---------------------------------------------------------------------------
# Google Drive — the one place work survives the session ending.
# ---------------------------------------------------------------------------
#
# Kaggle wipes /kaggle/working when a session ends or times out. Everything
# expensive — the dataset, the newest checkpoint, the final model — is written
# once locally and uploaded here. There is no second copy and no second
# cadence; the file you see on disk is the file that gets uploaded.
#
# One-time setup:
#   1. console.cloud.google.com -> new project -> enable the Drive API
#   2. Create a service account, then create a JSON key for it
#   3. In Drive, make a folder and share it with the service account's
#      client_email (found in the JSON) as Editor.
#      A service account has its own Drive with ZERO quota, so it can only
#      write into a folder you have shared with it, where the bytes count
#      against your quota. This is the step people miss, and it fails with a
#      confusing quota error rather than a permission error.
#   4. The folder id is the last part of
#      drive.google.com/drive/folders/<THIS_PART>
#   5. Upload the JSON key to Kaggle as a PRIVATE dataset, then point at it.
#      Keep it private: that key can write to the folder you shared.

DRIVE_FOLDER_ID = ""   # e.g. "1AbC2dEfGhIjKlMnOpQrStUvWxYz"
DRIVE_KEY_PATH  = ""   # e.g. "/kaggle/input/gdrive-key/service_account.json"

import os

if DRIVE_FOLDER_ID and DRIVE_KEY_PATH:
    os.environ["DRIVE_FOLDER_ID"] = DRIVE_FOLDER_ID
    os.environ["DRIVE_SERVICE_ACCOUNT_JSON"] = DRIVE_KEY_PATH

from lib.store import build_store
STORE = build_store()

if STORE.__class__.__name__ == "NullStore":
    print("")
    print("Nothing will survive this session ending. Fine for a smoke test;")
    print("fill in the two values above before starting a real run.")
else:
    # Prove the credentials work now, rather than discovering they do not
    # eight hours in when the first checkpoint tries to upload.
    from pathlib import Path
    probe = Path("/kaggle/working/.drive_probe")
    probe.write_text("ok")
    if STORE.push(probe, "_probe.txt") and STORE.pull("_probe.txt", probe):
        print("Drive write + read verified.")
    else:
        print("Drive is NOT working. Check that the folder is shared with the")
        print("service account's client_email as Editor.")
    probe.unlink(missing_ok=True)


In [ ]:
from pathlib import Path
from lib.data import SampleSpec, build_dataset, feature_statistics

SPEC = SampleSpec(total=SAMPLE_ROWS, adversarial_share=0.15, human_share=0.5)

# Reuses a local Parquet if present, else pulls it from the store, else builds
# it. Re-running this cell after a wiped session costs seconds, not hours.
frame_a = build_dataset(DATA / "train_a.parquet", SPEC,
                        include_adversarial=True, store=STORE)


In [ ]:
spec_b = SampleSpec(total=SAMPLE_ROWS, adversarial_share=0.0, human_share=0.5,
                    seed=SPEC.seed + 1)
frame_b = build_dataset(DATA / "train_b.parquet", spec_b,
                        include_adversarial=False, store=STORE)


In [ ]:
# Gate: class balance and domain coverage must be verified before training
# (PRD 18, phase 1). A skewed sample produces a model that looks fine on its
# own validation split and fails on everything else.
#
# The sampler already balances at selection time, but human text is the
# scarce class by an order of magnitude and the exact mix depends on what the
# stream happened to yield. This is the safety net: it downsamples the
# majority class within each adversarial group, so both the label balance and
# the adversarial share hold. A no-op when the frames are already balanced.
import pandas as pd

def rebalance(frame, name, seed=0):
    parts = []
    for is_adv, group in frame.groupby("adversarial", sort=False):
        humans = group[group["label"] == 0]
        ais = group[group["label"] == 1]
        n = min(len(humans), len(ais))
        tag = "adversarial" if is_adv else "clean"
        if n == 0:
            print(f"  [{name}/{tag}] one class is empty — kept unchanged")
            parts.append(group)
            continue
        if len(humans) != len(ais):
            print(f"  [{name}/{tag}] {len(humans):,} human / {len(ais):,} AI -> {n:,} each")
        parts.append(pd.concat([humans.sample(n=n, random_state=seed),
                                ais.sample(n=n, random_state=seed)]))
    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    if len(out) != len(frame):
        print(f"  [{name}] {len(frame):,} @ {frame['label'].mean():.3f} -> "
              f"{len(out):,} @ {out['label'].mean():.3f}")
    return out

frame_a = rebalance(frame_a, "A")
frame_b = rebalance(frame_b, "B")

# Stages 2 and 3 read the Parquet files, not these variables — write them back.
frame_a.to_parquet(DATA / "train_a.parquet", index=False)
frame_b.to_parquet(DATA / "train_b.parquet", index=False)

for name, frame in (("A", frame_a), ("B", frame_b)):
    print(f"--- Model {name}: {len(frame):,} rows")
    print(frame["label"].value_counts(normalize=True).round(3).to_dict())
    print("  generators:", frame["generator"].nunique(), " domains:", frame["domain"].nunique())
    print("  adversarial share:", round(frame["adversarial"].mean(), 3))
    print("  words: median", int(frame["text"].str.split().str.len().median()))

balance = frame_a["label"].mean()
assert 0.4 < balance < 0.6, f"Model A is unbalanced: {balance:.3f} positive"
assert frame_b["adversarial"].sum() == 0, "Model B must contain no adversarial rows"
print("\nGate passed.")


In [ ]:
# Feature standardisation statistics.
#
# Copy these into api/_lib/features.py (FEATURE_MEAN / FEATURE_STD) and commit
# before deploying. The values shipped there are rough placeholders; leaving
# them means inference standardises with different numbers than training used,
# which costs real accuracy. Skip this on a smoke test.
import json
stats = feature_statistics(frame_a)
(MODELS / "feature_stats.json").write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2)[:900])
